Clean the data and check if there is duplicate 

In [136]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"   # silences a harmless joblib warning on Windows

import pandas as pd
from pathlib import Path

# paths are relative to this notebook's folder (notebooks/), so run the notebook from there

data_dir = Path("../Datasets/Premier_League_Team_Statistics_2021-2026")

frames = []
for f in sorted(data_dir.glob("season-*.csv")):
    df = pd.read_csv(f)
    df["Season"] = f.stem          # e.g. season-2122
    frames.append(df)

matches = pd.concat(frames, ignore_index=True)
# the CSV dates are yyyy-mm-dd, so give that format explicitly (dayfirst=True would swap day and month in newer pandas)
matches["Date"] = pd.to_datetime(matches["Date"], format="%Y-%m-%d")
matches = matches.sort_values("Date").reset_index(drop=True)

print(matches.shape)
matches.head()

(3800, 23)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,2016-08-13,Burnley,Swansea,0,1,A,0,0,D,J Moss,...,9,10,14,7,4,3,2,0,0,season-1617
1,2016-08-13,Crystal Palace,West Brom,0,1,A,0,0,D,C Pawson,...,3,12,15,3,6,2,2,0,0,season-1617
2,2016-08-13,Everton,Tottenham,1,1,D,1,0,H,M Atkinson,...,4,10,14,5,6,0,0,0,0,season-1617
3,2016-08-13,Hull,Leicester,2,1,H,1,0,H,M Dean,...,5,8,17,5,3,2,2,0,0,season-1617
4,2016-08-13,Man City,Sunderland,2,1,H,1,0,H,R Madley,...,3,11,14,9,6,1,2,0,0,season-1617


In [137]:
import numpy as np

# 1. Missing values (we want all zeros)
print(matches.isna().sum())

# 2. Duplicate matches (we want 0)
print("duplicates:", matches.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum())

# 3. Rows per season (we want 380 each)
print(matches["Season"].value_counts().sort_index())

# 4. Team names (about 25-30 unique, no spelling variations)
print(sorted(matches["HomeTeam"].unique()))

# 5. Does FTR agree with the goals? (we want True)
expected = np.select(
    [matches["FTHG"] > matches["FTAG"], matches["FTHG"] < matches["FTAG"]],
    ["H", "A"],
    default="D",
)
print("FTR consistent:", (expected == matches["FTR"]).all())

# 6. Baseline: how often does each result happen?
print(matches["FTR"].value_counts(normalize=True).round(3))

Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HF          0
AF          0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
Season      0
dtype: int64
duplicates: 0
Season
season-1617    380
season-1718    380
season-1819    380
season-1920    380
season-2021    380
season-2122    380
season-2223    380
season-2324    380
season-2425    380
season-2526    380
Name: count, dtype: int64
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves'

In [138]:
rename_map = {
    "HomeTeam": "Home_Team",
    "AwayTeam": "Away_Team",
    "FTHG": "Home_Goals",
    "FTAG": "Away_Goals",
    "FTR": "Full_Time_Result",
    "HTHG": "Half_Time_Home_Goals",
    "HTAG": "Half_Time_Away_Goals",
    "HTR": "Half_Time_Result",
    "HS": "Home_Shots",
    "AS": "Away_Shots",
    "HST": "Home_Shots_On_Target",
    "AST": "Away_Shots_On_Target",
    "HF": "Home_Fouls",
    "AF": "Away_Fouls",
    "HC": "Home_Corners",
    "AC": "Away_Corners",
    "HY": "Home_Yellow_Cards",
    "AY": "Away_Yellow_Cards",
    "HR": "Home_Red_Cards",
    "AR": "Away_Red_Cards",
}

matches = matches.rename(columns=rename_map)
print(matches.columns.tolist())

# Known typo in the source file: West Ham had 9 shots on target but only 8 shots (Newcastle v West Ham, 2021-08-15)
matches.loc[(matches["Date"] == "2021-08-15") & (matches["Home_Team"] == "Newcastle"), "Away_Shots"] = 9

['Date', 'Home_Team', 'Away_Team', 'Home_Goals', 'Away_Goals', 'Full_Time_Result', 'Half_Time_Home_Goals', 'Half_Time_Away_Goals', 'Half_Time_Result', 'Referee', 'Home_Shots', 'Away_Shots', 'Home_Shots_On_Target', 'Away_Shots_On_Target', 'Home_Fouls', 'Away_Fouls', 'Home_Corners', 'Away_Corners', 'Home_Yellow_Cards', 'Away_Yellow_Cards', 'Home_Red_Cards', 'Away_Red_Cards', 'Season']


In [139]:
import numpy as np

matches["Match_ID"] = matches.index   # so we can join back later

# Team view: home side
home = matches.rename(columns={
    "Home_Team": "Team", "Away_Team": "Opponent",
    "Home_Goals": "Goals_For", "Away_Goals": "Goals_Against",
    "Home_Shots": "Shots", "Home_Shots_On_Target": "Shots_On_Target",
    "Home_Corners": "Corners",
})[["Match_ID", "Date", "Team", "Goals_For", "Goals_Against", "Shots", "Shots_On_Target", "Corners"]].copy()
home["Is_Home"] = 1

# Team view: away side
away = matches.rename(columns={
    "Away_Team": "Team", "Home_Team": "Opponent",
    "Away_Goals": "Goals_For", "Home_Goals": "Goals_Against",
    "Away_Shots": "Shots", "Away_Shots_On_Target": "Shots_On_Target",
    "Away_Corners": "Corners",
})[["Match_ID", "Date", "Team", "Goals_For", "Goals_Against", "Shots", "Shots_On_Target", "Corners"]].copy()
away["Is_Home"] = 0

team_games = pd.concat([home, away]).sort_values(["Team", "Date"]).reset_index(drop=True)

# Points earned in each match
team_games["Points"] = np.select(
    [team_games["Goals_For"] > team_games["Goals_Against"],
     team_games["Goals_For"] < team_games["Goals_Against"]],
    [3, 0], default=1)

# Rolling averages of the previous 5 matches
stats = ["Goals_For", "Goals_Against", "Shots", "Shots_On_Target", "Corners", "Points"]
for s in stats:
    team_games[f"{s}_Last5"] = (
        team_games.groupby("Team")[s].transform(lambda x: x.shift(1).rolling(5).mean())
    )

# Back to one row per match
feat_cols = [f"{s}_Last5" for s in stats]
h = team_games[team_games["Is_Home"] == 1].set_index("Match_ID")[feat_cols].add_prefix("Home_")
a = team_games[team_games["Is_Home"] == 0].set_index("Match_ID")[feat_cols].add_prefix("Away_")

model_df = matches.set_index("Match_ID").join(h).join(a)
model_df = model_df[["Date", "Season", "Home_Team", "Away_Team",
                     "Home_Goals", "Away_Goals", "Full_Time_Result"]
                    + list(h.columns) + list(a.columns)]

print(model_df.shape)
print(model_df.isna().sum().sum(), "missing values")
model_df.head(3)

(3800, 19)
1020 missing values


,Date,Season,Home_Team,Away_Team,Home_Goals,Away_Goals,Full_Time_Result,Home_Goals_For_Last5,Home_Goals_Against_Last5,Home_Shots_Last5,Home_Shots_On_Target_Last5,Home_Corners_Last5,Home_Points_Last5,Away_Goals_For_Last5,Away_Goals_Against_Last5,Away_Shots_Last5,Away_Shots_On_Target_Last5,Away_Corners_Last5,Away_Points_Last5
Match_ID,,,,,,,,,,,,,,,,,,,
0,2016-08-13,season-1617,Burnley,Swansea,0,1,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-08-13,season-1617,Crystal Palace,West Brom,0,1,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-08-13,season-1617,Everton,Tottenham,1,1,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [140]:
model_df = model_df.dropna().reset_index(drop=True)
print(model_df.shape)

(3681, 19)


In [141]:
print("before:", model_df.shape)
model_df = model_df.dropna().reset_index(drop=True)
print("after: ", model_df.shape)

before: (3681, 19)
after:  (3681, 19)


In [142]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

feature_cols = [c for c in model_df.columns if c.endswith("_Last5")]

# Split by date: train on older seasons, test on the newest one
train = model_df[model_df["Date"] < "2025-08-01"]
test  = model_df[model_df["Date"] >= "2025-08-01"]

X_train, y_train = train[feature_cols], train["Full_Time_Result"]
X_test,  y_test  = test[feature_cols],  test["Full_Time_Result"]

baseline = (y_test == "H").mean()

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))

print("train rows:", len(train), "| test rows:", len(test))
print(f"Baseline (always guess Home win): {baseline:.3f}")
print(f"Logistic Regression accuracy:     {acc:.3f}")

train rows: 3301 | test rows: 380
Baseline (always guess Home win): 0.426
Logistic Regression accuracy:     0.479


In [160]:
from sklearn.metrics import confusion_matrix, classification_report

pred = model.predict(X_test)
print(confusion_matrix(y_test, pred, labels=["H", "D", "A"]))
print(classification_report(y_test, pred, zero_division=0))

[[128   0  34]
 [ 64   0  40]
 [ 60   0  54]]
              precision    recall  f1-score   support

           A       0.42      0.47      0.45       114
           D       0.00      0.00      0.00       104
           H       0.51      0.79      0.62       162

    accuracy                           0.48       380
   macro avg       0.31      0.42      0.35       380
weighted avg       0.34      0.48      0.40       380



In [158]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error

home_model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))
away_model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))

home_model.fit(X_train, train["Home_Goals"])
away_model.fit(X_train, train["Away_Goals"])

test = test.copy()
test["Pred_Home_Goals"] = home_model.predict(X_test)
test["Pred_Away_Goals"] = away_model.predict(X_test)

base_h = train["Home_Goals"].mean()
base_a = train["Away_Goals"].mean()

print("Home goals error | model:", round(mean_absolute_error(test["Home_Goals"], test["Pred_Home_Goals"]), 3),
      "| baseline:", round(mean_absolute_error(test["Home_Goals"], [base_h]*len(test)), 3))
print("Away goals error | model:", round(mean_absolute_error(test["Away_Goals"], test["Pred_Away_Goals"]), 3),
      "| baseline:", round(mean_absolute_error(test["Away_Goals"], [base_a]*len(test)), 3))

test[["Home_Team", "Away_Team", "Home_Goals", "Away_Goals", "Pred_Home_Goals", "Pred_Away_Goals"]].head(10)

Home goals error | model: 0.97 | baseline: 0.997
Away goals error | model: 0.853 | baseline: 0.884


,Home_Team,Away_Team,Home_Goals,Away_Goals,Pred_Home_Goals,Pred_Away_Goals
3301,Liverpool,Bournemouth,4,2,1.753412,1.095113
3302,Tottenham,Burnley,3,0,1.056572,1.719354
3303,Sunderland,West Ham,3,0,1.325567,1.328490
3304,Wolves,Man City,0,4,1.268861,1.607931
3305,Aston Villa,Newcastle,0,0,1.237194,1.550538
3306,Brighton,Fulham,1,1,2.016017,1.086146
3307,Chelsea,Crystal Palace,0,0,1.523449,1.352272
3308,Nott'm Forest,Brentford,3,1,1.349594,1.408326
3309,Man United,Arsenal,0,1,1.547378,1.313256
3310,Leeds,Everton,1,0,1.276866,1.430780


In [159]:
home_wins = test["Pred_Home_Goals"] > test["Pred_Away_Goals"]
pred_from_goals = np.where(home_wins, "H", "A")
print("Accuracy from goals model:", (pred_from_goals == test["Full_Time_Result"]).mean().round(3))

Accuracy from goals model: 0.487


In [146]:
# Build Elo ratings from ALL matches, in date order
ratings = {}
K = 20          # how fast ratings move
HOME_ADV = 60   # extra rating points for the home team
rows = []

for _, r in matches.sort_values("Date").iterrows():
    h, a = r["Home_Team"], r["Away_Team"]
    rh, ra = ratings.get(h, 1500), ratings.get(a, 1500)   # new teams start at 1500

    rows.append((r["Date"], h, a, rh, ra))   # save ratings BEFORE the match (no leakage)

    expected_h = 1 / (1 + 10 ** (-((rh + HOME_ADV) - ra) / 400))
    if r["Home_Goals"] > r["Away_Goals"]:
        score_h = 1
    elif r["Home_Goals"] == r["Away_Goals"]:
        score_h = 0.5
    else:
        score_h = 0

    ratings[h] = rh + K * (score_h - expected_h)
    ratings[a] = ra - K * (score_h - expected_h)

elo_df = pd.DataFrame(rows, columns=["Date", "Home_Team", "Away_Team", "Home_Elo", "Away_Elo"])

# drop old Elo columns first, so re-running this cell does not create Home_Elo_x / Home_Elo_y copies
model_df = model_df.drop(columns=["Home_Elo", "Away_Elo", "Elo_Diff"], errors="ignore")
model_df = model_df.merge(elo_df, on=["Date", "Home_Team", "Away_Team"], how="left")
model_df["Elo_Diff"] = model_df["Home_Elo"] - model_df["Away_Elo"]
print(model_df.shape)   # should be (1815, 22)

(3681, 22)


In [156]:
form_cols = [c for c in model_df.columns if c.endswith("_Last5")]
elo_cols = ["Home_Elo", "Away_Elo", "Elo_Diff"]

train = model_df[model_df["Date"] < "2025-08-01"]
test  = model_df[model_df["Date"] >= "2025-08-01"]

for name, cols in [("form only", form_cols), ("Elo only", elo_cols), ("form + Elo", form_cols + elo_cols)]:
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(train[cols], train["Full_Time_Result"])
    acc = accuracy_score(test["Full_Time_Result"], m.predict(test[cols]))
    print(f"{name:12s} accuracy: {acc:.3f}")

form only    accuracy: 0.479
Elo only     accuracy: 0.484
form + Elo   accuracy: 0.492


In [148]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error

cols = form_cols + elo_cols
home_model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))
away_model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))
home_model.fit(train[cols], train["Home_Goals"])
away_model.fit(train[cols], train["Away_Goals"])

test = test.copy()
test["Pred_Home_Goals"] = home_model.predict(test[cols])
test["Pred_Away_Goals"] = away_model.predict(test[cols])

print("Home goals error | form only: 0.973 | with Elo:",
      round(mean_absolute_error(test["Home_Goals"], test["Pred_Home_Goals"]), 3))
print("Away goals error | form only: 0.870 | with Elo:",
      round(mean_absolute_error(test["Away_Goals"], test["Pred_Away_Goals"]), 3))

Home goals error | form only: 0.973 | with Elo: 0.945
Away goals error | form only: 0.870 | with Elo: 0.847


In [149]:
from pathlib import Path

out_dir = Path("../Datasets/processed")
out_dir.mkdir(exist_ok=True)
model_df.to_csv(out_dir / "matches_features_elo.csv", index=False)

In [150]:
print(len(model_df), model_df["Date"].min(), model_df["Date"].max())

3681 2016-09-24 00:00:00 2026-05-24 00:00:00


In [151]:
cols = form_cols + elo_cols
final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_model.fit(model_df[cols], model_df["Full_Time_Result"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some 

In [154]:
def predict_match(home, away):
    stats = ["Goals_For", "Goals_Against", "Shots", "Shots_On_Target", "Corners", "Points"]

    def last5(team):
        g = team_games[team_games["Team"] == team].sort_values("Date").tail(5)
        if g.empty:
            raise ValueError(f"Unknown team: {team!r}. Use the names in team_games['Team'].unique()")
        return g[stats].mean()

    h5, a5 = last5(home), last5(away)
    row = {}
    for s in stats:
        row[f"Home_{s}_Last5"] = h5[s]
        row[f"Away_{s}_Last5"] = a5[s]
    row["Home_Elo"] = ratings.get(home, 1500)
    row["Away_Elo"] = ratings.get(away, 1500)
    row["Elo_Diff"] = row["Home_Elo"] - row["Away_Elo"]

    X_new = pd.DataFrame([row])[cols]

    probs = final_model.predict_proba(X_new)[0]   # order: A, D, H
    print(f"{home} vs {away}")
    print(f"  {home} win: {probs[2]:.0%} | Draw: {probs[1]:.0%} | {away} win: {probs[0]:.0%}")
    print(f"  Predicted winner: {final_model.predict(X_new)[0]}  (H = home, A = away)")
    print(f"  Expected goals: {home_model.predict(X_new)[0]:.1f} - {away_model.predict(X_new)[0]:.1f}")

predict_match("Arsenal", "Chelsea")
predict_match("Liverpool", "Man City")

Arsenal vs Chelsea
  Arsenal win: 73% | Draw: 19% | Chelsea win: 8%
  Predicted winner: H  (H = home, A = away)
  Expected goals: 2.2 - 0.8
Liverpool vs Man City
  Liverpool win: 32% | Draw: 20% | Man City win: 48%
  Predicted winner: A  (H = home, A = away)
  Expected goals: 1.4 - 1.7


In [155]:
def season_table(df):
    home = pd.DataFrame({"Team": df["Home_Team"], "GF": df["Home_Goals"], "GA": df["Away_Goals"]})
    away = pd.DataFrame({"Team": df["Away_Team"], "GF": df["Away_Goals"], "GA": df["Home_Goals"]})
    t = pd.concat([home, away])
    t["W"] = (t["GF"] > t["GA"]).astype(int)
    t["D"] = (t["GF"] == t["GA"]).astype(int)
    t["L"] = (t["GF"] < t["GA"]).astype(int)
    t = t.groupby("Team").agg(P=("W", "size"), W=("W", "sum"), D=("D", "sum"),
                              L=("L", "sum"), GF=("GF", "sum"), GA=("GA", "sum"))
    t["GD"] = t["GF"] - t["GA"]
    t["Pts"] = 3 * t["W"] + t["D"]
    return t.sort_values(["Pts", "GD", "GF"], ascending=False)

for season, df in matches.groupby("Season"):
    print(season)
    print(season_table(df).head(3), "\n")

season-1617
            P   W  D  L  GF  GA  GD  Pts
Team                                    
Chelsea    38  30  3  5  85  33  52   93
Tottenham  38  26  8  4  86  26  60   86
Man City   38  23  9  6  80  39  41   78 

season-1718
             P   W  D  L   GF  GA  GD  Pts
Team                                      
Man City    38  32  4  2  106  27  79  100
Man United  38  25  6  7   68  28  40   81
Tottenham   38  23  8  7   74  36  38   77 

season-1819
            P   W  D  L  GF  GA  GD  Pts
Team                                    
Man City   38  32  2  4  95  23  72   98
Liverpool  38  30  7  1  89  22  67   97
Chelsea    38  21  9  8  63  39  24   72 

season-1920
             P   W   D  L   GF  GA  GD  Pts
Team                                       
Liverpool   38  32   3  3   85  33  52   99
Man City    38  26   3  9  102  35  67   81
Man United  38  18  12  8   66  36  30   66 

season-2021
             P   W   D  L  GF  GA  GD  Pts
Team                                      
M